# Feature Selection

## Learning objectives
- Use `SelectKBest` to select features.
- Use `SelectFromModel` for model-based selection.
- Avoid leakage by fitting selectors only on training data.

In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
print(X.shape)
X.head()

(569, 30)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 1. Select the top 10 features
`f_classif` scores each numeric feature against the class labels.

In [3]:
from sklearn.feature_selection import SelectKBest, f_classif
selector = SelectKBest(score_func=f_classif, k=10)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)
selected = X.columns[selector.get_support()]
print(selected.tolist())
pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False).head(10)

['mean radius', 'mean perimeter', 'mean area', 'mean concavity', 'mean concave points', 'worst radius', 'worst perimeter', 'worst area', 'worst concavity', 'worst concave points']


worst concave points    795.223341
worst perimeter         719.598823
mean concave points     703.364031
worst radius            687.430367
mean perimeter          565.485755
worst area              541.713694
mean radius             523.613907
mean area               469.516453
mean concavity          416.374874
worst concavity         365.747389
dtype: float64

## 2. Compare model performance
The selector is fitted on the training data within the pipeline.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

models = {
    "All features": Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=5000))]),
    "Top 10 features": Pipeline([("scale", StandardScaler()), ("select", SelectKBest(f_classif, k=10)), ("model", LogisticRegression(max_iter=5000))]),
}
for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, round(accuracy_score(y_test, model.predict(X_test)), 4))

All features 0.9825
Top 10 features 0.9474


## 3. Model-based selection
`SelectFromModel` keeps features whose learned importance meets the threshold.

In [5]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

selector_model = SelectFromModel(RandomForestClassifier(n_estimators=200, random_state=42), threshold="median")
selector_model.fit(X_train, y_train)
print(X.columns[selector_model.get_support()].tolist())

['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean compactness', 'mean concavity', 'mean concave points', 'area error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst compactness', 'worst concavity', 'worst concave points']


## Key takeaways
- Selection can reduce complexity, but does not guarantee better results.
- Fit selectors on training data only.
- Compare choices with a held-out test set or cross-validation.